### Wandertag [Bwinf43 2024](https://bwinf.de/bundeswettbewerb/43/) - A3

[Video](https://youtu.be/NqzUbMcmQL8)

<img src='wandertag.png' width='300'>

#### Beispieldaten

Jede Datei enthält Streckenwünsche und ist so aufgebaut:

- Zeile 1: Anzahl n der Personen mit Streckenwünschen.
- In den folgenden n Zeilen ist jeweils die kürzeste und längste gewünschte Strecke aus Sicht einer Person angegeben (in Metern). Die angegebenen minimalen und maximalen Streckenlängen sind inklusive, also gilt [x_1, x_2].


Hier ist eine Beispieleingabe:

    3
    500 1000
    100 200
    100 600

In diesem Beispiel gibt es insgesamt drei Streckenwünsche. Person 1 möchte gern mindestens 500m und maximal 1000m laufen. 
Sollte die angebotene Strecke genau 500m oder 1000m betragen, würde sie auch mitkommen.
Analog wünscht sich Person 2 eine Strecke zwischen 100m und 200m und Person 3 zwischen 100m und 600m.

Für die Dateien wandern6.txt und wandern7.txt benötigt dein Programm eventuell mehr Zeit als für die anderen Eingaben. Versuche, diese Zeit zu minimieren.

In [ ]:
# Anschauen der Beispieldaten 1-7
nr = 1
print(f'Beispiel {nr}:')
f = open(f'beispieldaten/wandern{nr}.txt')
print(f.read().strip())
f.close()

#### Einlesen der Daten

In [ ]:
nr  = '3'
f = open('./beispieldaten/wandern'+nr+'.txt')
n = int(f.readline())
data = []
for i in range(n):
    x, y = f.readline().split()
    data.append((int(x),int(y)))
f.close()
print(f'Beispiel: {nr}')
print(f'Anzahl Personen: {n}')
print('Streckenwünsche:')
for x, y in data[:30]:
    print(f'{x:5} - {y:5}')


#### Überlegungen

Bei Intervallbeginnen können Teilnehmer dazukommen.
Wie auch immer eine optimale Lösung für das Problem aussieht, es gibt dazu auch eine optimale Lösung mit drei Intervallbeginnen.

In [ ]:
import math
anfang = sorted({x[0] for x in data}) 
print(f'Anzahl Intervallbeginne: {len(anfang)}')
print(f'Anzahl 3er-Kombinationen von Intervallbeginnen: {math.comb(len(anfang),3)}')
print('Die sortierte Menge der Intervallbeginne (höchstens 30):')
print(anfang[:30])

#### Brute Force 
Für die kleinen Eingabedaten machen wir ein brute-force, um die Lösungen für andere Verfahren zu vergleichen.

In [ ]:
def evaluate(x,y,z):
    zaehl = 0
    for von,bis in data:
        if von <= x <= bis or von <= y <= bis or von <= z <= bis:
            zaehl+=1
    return zaehl

In [ ]:
import itertools as it
best = 0
bestVal = None
for x,y,z in it.combinations(anfang,3):
    val = evaluate(x,y,z)
    if bestVal is None or val > bestVal:
        bestVal = val
        best = x,y,z

print(f'Die drei Streckenlängen:',best)
print(f'Anzahl Teilnehmer: {bestVal} von {n}.')

#### Brute force insgesamt

In [30]:
%%time
import itertools as it
def evaluate(x,y,z):
    zaehl = 0
    for von,bis in data:
        if von <= x <= bis or von <= y <= bis or von <= z <= bis:
            zaehl+=1
    return zaehl
    
nr  = '3'
f = open('./beispieldaten/wandern'+nr+'.txt')
n = int(f.readline())
data = []
for i in range(n):
    x, y = f.readline().split()
    data.append((int(x),int(y)))
f.close()
print(f'Beispiel: {nr}')

anfang = sorted({x[0] for x in data}) 

best = 0
bestVal = None
for x,y,z in it.combinations(anfang,3):
    val = evaluate(x,y,z)
    if bestVal is None or val > bestVal:
        bestVal = val
        best = x,y,z

print(f'Die drei Streckenlängen:',best)
print(f'Anzahl Teilnehmer: {bestVal} von {n}.')

Beispiel: 3
Die drei Streckenlängen: (19, 66, 92)
Anzahl Teilnehmer: 10 von 10.
CPU times: total: 0 ns
Wall time: 2.54 ms


#### Weitere Überlegungen

Teilnehmerzahlen können ändern sich, wenn der Beginn eines Intervalls gewählt wird, oder die nächste Zahl nach dem Intervallende. Wir bezeichnen diese Zahlen als kritische Längen.

In [33]:
kritisch = sorted({x[0] for x in data} | {x[1]+1 for x in data}) 

In [34]:
kritisch

[16, 17, 19, 23, 26, 27, 28, 53, 66, 68, 73, 89, 90, 91, 92, 95, 97]

Wir ordnen jeder kritischen Zahl zu, wie sich bei ihr die Teilnehmerzahl ändert.

In [35]:
diff = {x:0 for x in kritisch}
for von, bis in data:
    diff[von]+=1
    diff[bis+1]-=1
 

In [36]:
diff

{16: 1,
 17: 1,
 19: 1,
 23: -1,
 26: 1,
 27: 1,
 28: -1,
 53: 1,
 66: 0,
 68: -2,
 73: -1,
 89: 1,
 90: 1,
 91: -1,
 92: 1,
 95: -2,
 97: -1}

Wir erzeugen uns eine Liste *anz* mit der Anzahl der Teilnehmer für jede kritische Länge.


In [37]:
anz = []
val = 0
for x in kritisch:
    val += diff[x]
    anz.append(val)
print(anz)

[1, 2, 3, 2, 3, 4, 3, 4, 4, 2, 1, 2, 3, 2, 3, 1, 0]


In [39]:
maxkritisch = []
for i in range(1,len(kritisch)-1):
    if anz[i-1] <= anz[i] >= anz[i+1]:
        maxkritisch.append(kritisch[i])

print(maxkritisch)


[19, 27, 53, 66, 90, 92]


Wir überprüfen nur 3er Kombinationen von kritischen Längen, die in der Liste anz ein lokales Maximum haben.

In [40]:
maxkritisch = []
for i in range(1,len(kritisch)-1):
    if anz[i-1] <= anz[i] >= anz[i+1]:
        maxkritisch.append(kritisch[i])

if anz[0] >= anz[1]:
    maxkritisch.append(kritisch[0])
if anz[-1] >= anz[-2]:
    maxkritisch.append(kritisch[-1])

print(f'Anzahl maxkritischer Längen: {len(maxkritisch)}')
print(f'Anzahl 3er-Kombinationen mit maxkritischen Längen: {math.comb(len(maxkritisch),3)}')
print('Die sortierte Menge der maxkritischer Längen (höchstens 30):')
print(maxkritisch[:30])

Anzahl maxkritischer Längen: 6
Anzahl 3er-Kombinationen mit maxkritischen Längen: 20
Die sortierte Menge der maxkritischer Längen (höchstens 30):
[19, 27, 53, 66, 90, 92]


#### Brute force

Diesmal nur für die maxkritischen 3er-Kombinationen

In [45]:
%%time
import math
import itertools as it

def evaluate(x,y,z):
    zaehl = 0
    for von,bis in data:
        if von <= x <= bis or von <= y <= bis or von <= z <= bis:
            zaehl+=1
    return zaehl
    
nr  = '7'
f = open('./beispieldaten/wandern'+nr+'.txt')
n = int(f.readline())
data = []
for i in range(n):
    x, y = f.readline().strip().split()
    data.append((int(x),int(y)))
f.close()

print(f'Beispiel: {nr}')
print(f'Anzahl Personen: {n}')

kritisch = sorted({x[0] for x in data} | {x[1]+1 for x in data}) 

# Differenzen bei den kritischen Längen
diff = {x:0 for x in kritisch}
for von, bis in data:
    diff[von]+=1
    diff[bis+1]-=1

# Anzahl Teilnehmer bei den kritischen Längen
anz = []
val = 0
for x in kritisch:
    val += diff[x]
    anz.append(val)
 
maxkritisch = []
for i in range(1,len(kritisch)-1):
    if anz[i-1] <= anz[i] >= anz[i+1]:
        maxkritisch.append(kritisch[i])

if anz[0] >= anz[1]:
    maxkritisch.append(kritisch[0])
if anz[-1] >= anz[-2]:
    maxkritisch.append(kritisch[-1])

print(f'Anzahl maxkritischer Längen: {len(maxkritisch)}')
print(f'Anzahl 3er-Kombinationen mit maxkritischen Längen: {math.comb(len(maxkritisch),3)}')

# brute force nach der besten Lösung suchen
best = 0
bestVal = None
for x,y,z in it.combinations(maxkritisch,3):
    val = evaluate(x,y,z)
    if bestVal is None or val > bestVal:
        bestVal = val
        best = x,y,z

print(f'Die drei Streckenlängen:',best)
print(f'Anzahl Teilnehmer: {bestVal} von {n}.')


Beispiel: 7
Anzahl Personen: 800
Anzahl maxkritischer Längen: 330
Anzahl 3er-Kombinationen mit maxkritischen Längen: 5935160
Die drei Streckenlängen: (39520, 76088, 91584)
Anzahl Teilnehmer: 551 von 800.
CPU times: total: 7min 57s
Wall time: 7min 59s


In [ ]:
m = dict()
for i in range(len(kritisch)):
    m[kritisch[i]] = anz[i]

for x in maxkritisch:
    print(x,m[x])

In [49]:
maxmaxkritisch = []
for i in range(1,len(maxkritisch)-1):
    vorher = m[maxkritisch[i-1]]
    current = m[maxkritisch[i]]
    nachher = m[maxkritisch[i+1]]
        
    if vorher <= current >= nachher:
        maxmaxkritisch.append(maxkritisch[i])

len(maxmaxkritisch)

120

Zur Verbesserung der Laufzeit wählen wir für die Fälle, wo wir mehr als 1 Million maxkritische Längen haben, nur solche maxkritischen Längen aus, die wiederum lokale Maxima in der Reihe der maxkritischen Teilnehmerzahlen aufweisen.

In [50]:
%%time
# brute force nach der besten Lösung suchen
best = 0
bestVal = None
for x,y,z in it.combinations(maxmaxkritisch,3):
    val = evaluate(x,y,z)
    if bestVal is None or val > bestVal:
        bestVal = val
        best = x,y,z

print(f'Die drei Streckenlängen:',best)
print(f'Anzahl Teilnehmer: {bestVal} von {n}.')

Die drei Streckenlängen: (39520, 76088, 91584)
Anzahl Teilnehmer: 551 von 800.
CPU times: total: 20.6 s
Wall time: 20.6 s
